# Capstone – Gestión de Telecomunicaciones II  
## Proyecto 1: 5G FWA Rural (Escenario Base Uniforme)

Dataset y escenario numérico **uniforme** para todos los grupos.

**Archivos (carpeta `data/`):**
- `demand_points.csv` (puntos de demanda con hogares y suscriptores iniciales)
- `candidate_sites.csv` (sitios candidatos)
- `link_budget_matrices.npz` (dist_km, pathloss_db, rsrp_dbm)
- `scenario_params.json` (parámetros del escenario)

Incluye:
- Modelo simplificado de SINR (reuse-1) y capacidad
- Baseline de selección de sitios
- Plantilla de búsqueda local para Sprint 4


In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
demand = pd.read_csv(DATA_DIR/"demand_points.csv")
sites = pd.read_csv(DATA_DIR/"candidate_sites.csv")
mat = np.load(DATA_DIR/"link_budget_matrices.npz")
dist_km = mat["dist_km"]
pathloss_db = mat["pathloss_db"]
rsrp_dbm = mat["rsrp_dbm"]
params = json.loads((DATA_DIR/"scenario_params.json").read_text())

print("Demand points:", len(demand))
print("Candidate sites:", len(sites))
print("RSRP matrix:", rsrp_dbm.shape)
demand.head()


Demand points: 220
Candidate sites: 36
RSRP matrix: (220, 36)


,dp_id,x_km,y_km,settlement_type,landclass,households,subscribers_initial
0,0,3.087773,3.081281,village,forest,14,4
1,1,4.950802,3.164556,village,forest,17,5
2,2,2.948100,5.386985,village,forest,15,4
3,3,4.610908,2.772241,village,rural,16,5
4,4,4.898507,3.167570,village,forest,12,4


In [4]:
BW_HZ = params["radio"]["bandwidth_mhz"]*1e6
NF_DB = params["radio"]["noise_figure_db"]
SE_CAP = 6.0

def dbm_to_w(dbm):
    return 10**((dbm-30)/10)

def thermal_noise_dbm(bw_hz, nf_db):
    return -174 + 10*np.log10(bw_hz) + nf_db

NOISE_DBM = thermal_noise_dbm(BW_HZ, NF_DB)
noise_w = dbm_to_w(NOISE_DBM)
print("Noise (dBm):", NOISE_DBM)


Noise (dBm): -90.97940008672037


In [5]:
def evaluate_plan(selected_sites, reuse1=True):
    sel = np.array(selected_sites, dtype=int)
    if len(sel)==0:
        return None

    rsrp_sel = rsrp_dbm[:, sel]  # [DP,S]
    best_local = np.argmax(rsrp_sel, axis=1)
    Pr_serv_dbm = rsrp_sel[np.arange(rsrp_sel.shape[0]), best_local]
    Pr_serv_w = dbm_to_w(Pr_serv_dbm)

    if reuse1 and len(sel)>1:
        Pr_all_w = dbm_to_w(rsrp_sel)
        interf_w = Pr_all_w.sum(axis=1) - Pr_serv_w
    else:
        interf_w = np.zeros_like(Pr_serv_w)

    sinr = Pr_serv_w / (interf_w + noise_w)
    sinr_db = 10*np.log10(sinr + 1e-12)

    se = np.log2(1+sinr)
    se = np.minimum(se, SE_CAP)
    cap_bps = BW_HZ * se

    subs = demand["subscribers_initial"].to_numpy()
    conc = params["demand"]["busy_hour_concurrency"]
    target_mbps = params["demand"]["target_throughput_mbps_per_subscriber"]
    demand_bps = subs * conc * target_mbps * 1e6

    satisfied = cap_bps >= demand_bps
    hh = demand["households"].to_numpy()

    pop_cov = float((hh*(cap_bps>0)).sum()/hh.sum())
    sat_frac = float((hh*satisfied).sum()/hh.sum())

    metrics = {
        "sites_deployed": int(len(sel)),
        "pop_covered_frac": pop_cov,
        "capacity_satisfied_frac": sat_frac,
        "avg_sinr_db": float(np.mean(sinr_db)),
        "p05_sinr_db": float(np.quantile(sinr_db, 0.05)),
        "avg_se_bphz": float(np.mean(se)),
    }
    return metrics

# Example
evaluate_plan([0,1,2,3,4])


{'sites_deployed': 5,
 'pop_covered_frac': 1.0,
 'capacity_satisfied_frac': 0.7,
 'avg_sinr_db': -0.9241238832473755,
 'p05_sinr_db': -24.12337312698364,
 'avg_se_bphz': 1.8885685205459595}

In [6]:
def greedy_by_best_rsrp(S=8, thr_dbm=-95):
    score = (rsrp_dbm > thr_dbm).sum(axis=0)
    return list(np.argsort(-score)[:S])

def objective(m, w_cov=1.0, w_cap=1.0, w_sites=0.15):
    return w_cov*m["pop_covered_frac"] + w_cap*m["capacity_satisfied_frac"] - w_sites*(m["sites_deployed"]/params["planning"]["max_sites_to_deploy"])

def local_search(initial_sites, iters=400, S=8, seed=0):
    rng = np.random.default_rng(seed)
    all_sites = np.arange(len(sites))

    best = sorted(list(dict.fromkeys(initial_sites)))[:S]
    best_m = evaluate_plan(best)
    best_J = objective(best_m)

    for _ in range(iters):
        cand = best.copy()
        out = int(rng.integers(0, S))
        cand[out] = int(rng.choice(all_sites))
        cand = sorted(list(set(cand)))
        if len(cand)>S:
            cand = cand[:S]
        while len(cand)<S:
            cand.append(int(rng.choice(all_sites)))
            cand = sorted(list(set(cand)))
            if len(cand)>S:
                cand = cand[:S]

        m = evaluate_plan(cand)
        J = objective(m)
        if J > best_J:
            best, best_m, best_J = cand, m, J

    return best, best_m, best_J

init = greedy_by_best_rsrp(S=8)
best_sites, best_m, best_J = local_search(init, iters=600, S=8, seed=1)
print("Init:", init, evaluate_plan(init))
print("Best:", best_sites, best_m, best_J)


Init: [3, 2, 0, 7, 4, 16, 6, 11] {'sites_deployed': 8, 'pop_covered_frac': 1.0, 'capacity_satisfied_frac': 0.9583333333333334, 'avg_sinr_db': 6.540907859802246, 'p05_sinr_db': -10.55547342300415, 'avg_se_bphz': 2.5770325660705566}
Best: [0, 3, 6, 7, 11, 12, 13, 16] {'sites_deployed': 8, 'pop_covered_frac': 1.0, 'capacity_satisfied_frac': 0.9795833333333334, 'avg_sinr_db': 7.907271862030029, 'p05_sinr_db': -9.585938787460327, 'avg_se_bphz': 2.8853871822357178} 1.8795833333333332


# SPRINT 2 — MODELO TÉCNICO COMPLETO: SINR Y CAPACIDAD
## Objetivo del Sprint 2

En este sprint se revisa, implementa y valida de manera explícita el modelo técnico de la red 5G FWA.

La cadena de cálculo que se estudiará es:

**Sitios desplegados → Asociación por mejor RSRP → Interferencia + Ruido → SINR → Eficiencia espectral → Capacidad → Demanda Busy Hour → Cobertura y satisfacción de capacidad → Métricas**

El código original del notebook base se conserva sin modificaciones como referencia del Sprint 1.  
Las siguientes secciones corresponden al desarrollo y validación realizados por el equipo durante el Sprint 2.

---

## Asociación de cada punto de demanda por mejor RSRP

In [7]:
def associate_by_best_rsrp(selected_sites):
    """
    Asocia cada punto de demanda con el sitio desplegado
    que presenta el mayor RSRP.

    Parametros
    ----------
    selected_sites : lista
        Indices de los sitios que se consideran desplegados.

    Retorna
    -------
    rsrp_selected : matriz
        RSRP entre cada punto de demanda y los sitios desplegados.

    serving_site_local : vector
        Posicion del sitio servidor dentro de selected_sites.

    serving_site_global : vector
        Indice real del sitio servidor dentro de candidate_sites.csv.

    serving_rsrp_dbm : vector
        Mejor RSRP recibido por cada punto de demanda.
    """

    selected_sites = np.array(selected_sites, dtype=int)

    # Verificar que exista al menos un sitio desplegado
    if len(selected_sites) == 0:
        raise ValueError("Debe existir al menos un sitio desplegado.")

    # Extraer unicamente las columnas correspondientes
    # a los sitios que estan desplegados
    rsrp_selected = rsrp_dbm[:, selected_sites]

    # Para cada punto de demanda se busca el mayor RSRP.
    # El resultado indica la posicion dentro de selected_sites.
    serving_site_local = np.argmax(rsrp_selected, axis=1)

    # Convertir la posicion local al indice real del sitio candidato
    serving_site_global = selected_sites[serving_site_local]

    # Obtener el valor de RSRP correspondiente al sitio servidor
    serving_rsrp_dbm = rsrp_selected[
        np.arange(len(demand)),
        serving_site_local
    ]

    return (
        rsrp_selected,
        serving_site_local,
        serving_site_global,
        serving_rsrp_dbm
    )


# Evaluacion usando los mejores sitios obtenidos anteriormente
# en el notebook base

selected_sites_s2 = best_sites

(
    rsrp_selected_s2,
    serving_site_local_s2,
    serving_site_global_s2,
    serving_rsrp_dbm_s2
) = associate_by_best_rsrp(selected_sites_s2)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

# Debe existir una asociacion para cada punto de demanda
assert len(serving_site_global_s2) == len(demand)

# Todos los sitios servidores deben pertenecer al conjunto desplegado
assert set(np.unique(serving_site_global_s2)).issubset(
    set(selected_sites_s2)
)

# El RSRP servidor debe ser exactamente el maximo RSRP
# entre los sitios desplegados para cada punto
assert np.allclose(
    serving_rsrp_dbm_s2,
    np.max(rsrp_selected_s2, axis=1)
)

print("PB13 - Asociacion por mejor RSRP")
print("----------------------------------")
print("Sitios desplegados:", selected_sites_s2)
print("Puntos de demanda evaluados:", len(demand))
print("Todos los puntos tienen sitio servidor: OK")
print("Todos los sitios servidores pertenecen al plan desplegado: OK")
print("El RSRP servidor corresponde al maximo disponible: OK")


# Mostrar algunos resultados
association_results = pd.DataFrame({
    "demand_point": np.arange(len(demand)),
    "serving_site": serving_site_global_s2,
    "serving_rsrp_dbm": serving_rsrp_dbm_s2
})

association_results.head(10)

PB13 - Asociacion por mejor RSRP
----------------------------------
Sitios desplegados: [0, 3, 6, 7, 11, 12, 13, 16]
Puntos de demanda evaluados: 220
Todos los puntos tienen sitio servidor: OK
Todos los sitios servidores pertenecen al plan desplegado: OK
El RSRP servidor corresponde al maximo disponible: OK


,demand_point,serving_site,serving_rsrp_dbm
0,0,0,-75.471771
1,1,3,-67.153893
2,2,16,-91.421471
3,3,3,-72.603554
4,4,3,-70.460770
5,5,3,-80.579575
6,6,3,-85.704315
7,7,3,-84.480751
8,8,3,-65.780739
9,9,3,-76.025604


## Cálculo de interferencia reuse-1

In [8]:
def calculate_reuse1_interference(rsrp_selected, serving_rsrp_dbm):
    """
    Calcula la interferencia total recibida por cada punto
    de demanda bajo un esquema frequency reuse-1.

    Parámetros
    ----------
    rsrp_selected : matriz
        RSRP de cada punto de demanda respecto a los sitios desplegados.

    serving_rsrp_dbm : vector
        RSRP correspondiente al sitio servidor de cada punto.

    Retorna
    -------
    received_power_all_w : matriz
        Potencias recibidas desde todos los sitios desplegados,
        expresadas en watts.

    serving_power_w : vector
        Potencia recibida desde el sitio servidor, en watts.

    interference_w : vector
        Interferencia total recibida por cada punto, en watts.
    """

    # Convertir todas las potencias recibidas de dBm a watts
    received_power_all_w = dbm_to_w(rsrp_selected)

    # Convertir la potencia del sitio servidor de dBm a watts
    serving_power_w = dbm_to_w(serving_rsrp_dbm)

    # Sumar la potencia recibida desde todos los sitios desplegados
    total_received_power_w = np.sum(
        received_power_all_w,
        axis=1
    )

    # La interferencia es la potencia total menos la potencia servidora
    interference_w = (
        total_received_power_w
        - serving_power_w
    )

    return (
        received_power_all_w,
        serving_power_w,
        interference_w
    )


# ------------------------------------------------------------
# Calcular interferencia para el escenario obtenido en PB13
# ------------------------------------------------------------

(
    received_power_all_w_s2,
    serving_power_w_s2,
    interference_w_s2
) = calculate_reuse1_interference(
    rsrp_selected_s2,
    serving_rsrp_dbm_s2
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

# Debe existir un valor de interferencia por cada punto de demanda
assert len(interference_w_s2) == len(demand)

# La interferencia no puede ser negativa
assert np.all(interference_w_s2 >= -1e-15)

# Verificar que:
# potencia servidora + interferencia = potencia total recibida
assert np.allclose(
    serving_power_w_s2 + interference_w_s2,
    np.sum(received_power_all_w_s2, axis=1)
)

print("PB14 — Interferencia reuse-1")
print("----------------------------------")
print("Puntos de demanda evaluados:", len(demand))
print("Interferencia calculada para todos los puntos: OK")
print("La interferencia es no negativa: OK")
print("La potencia servidora fue excluida correctamente: OK")


# ------------------------------------------------------------
# Mostrar algunos resultados
# ------------------------------------------------------------

interference_results = pd.DataFrame({
    "demand_point": np.arange(len(demand)),
    "serving_site": serving_site_global_s2,
    "serving_rsrp_dbm": serving_rsrp_dbm_s2,
    "serving_power_w": serving_power_w_s2,
    "interference_w": interference_w_s2
})

interference_results.head(10)

PB14 — Interferencia reuse-1
----------------------------------
Puntos de demanda evaluados: 220
Interferencia calculada para todos los puntos: OK
La interferencia es no negativa: OK
La potencia servidora fue excluida correctamente: OK


,demand_point,serving_site,serving_rsrp_dbm,serving_power_w,interference_w
0,0,0,-75.471771,2.836761e-11,1.713305e-12
1,1,3,-67.153893,1.925798e-10,3.913064e-12
2,2,16,-91.421471,7.208633e-13,1.038675e-13
3,3,3,-72.603554,5.490918e-11,5.451837e-12
4,4,3,-70.460770,8.993386e-11,4.583584e-12
5,5,3,-80.579575,8.750691e-12,2.229999e-12
6,6,3,-85.704315,2.688861e-12,1.186722e-12
7,7,3,-84.480751,3.563893e-12,8.147916e-13
8,8,3,-65.780739,2.641962e-10,1.180307e-10
9,9,3,-76.025604,2.497123e-11,6.557326e-12


## Cálculo y validación del SINR por punto de demanda

In [9]:
def calculate_sinr(
    serving_power_w,
    interference_w,
    noise_power_w
):
    """
    Calcula el SINR de cada punto de demanda.

    Parámetros
    ----------
    serving_power_w : vector
        Potencia recibida desde el sitio servidor, en watts.

    interference_w : vector
        Interferencia total recibida desde los demás sitios,
        en watts.

    noise_power_w : float
        Potencia de ruido térmico, en watts.

    Retorna
    -------
    sinr_linear : vector
        SINR en escala lineal.

    sinr_db : vector
        SINR expresado en dB.
    """

    # SINR en escala lineal
    sinr_linear = serving_power_w / (
        interference_w + noise_power_w
    )

    # Conversión a dB
    sinr_db = 10 * np.log10(
        sinr_linear + 1e-12
    )

    return sinr_linear, sinr_db


# ------------------------------------------------------------
# Calcular SINR para los 220 puntos
# ------------------------------------------------------------

sinr_linear_s2, sinr_db_s2 = calculate_sinr(
    serving_power_w_s2,
    interference_w_s2,
    noise_w
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

# Debe existir un SINR por cada punto de demanda
assert len(sinr_db_s2) == len(demand)

# El SINR lineal debe ser positivo
assert np.all(sinr_linear_s2 > 0)

# Todos los resultados deben ser valores finitos
assert np.all(np.isfinite(sinr_linear_s2))
assert np.all(np.isfinite(sinr_db_s2))


# ------------------------------------------------------------
# Métricas básicas de SINR
# ------------------------------------------------------------

avg_sinr_db_s2 = np.mean(sinr_db_s2)
p05_sinr_db_s2 = np.quantile(sinr_db_s2, 0.05)


# Comparar con el resultado del baseline
baseline_metrics_s2 = evaluate_plan(selected_sites_s2)

assert np.isclose(
    avg_sinr_db_s2,
    baseline_metrics_s2["avg_sinr_db"]
)

assert np.isclose(
    p05_sinr_db_s2,
    baseline_metrics_s2["p05_sinr_db"]
)


print("PB15 — Cálculo de SINR")
print("----------------------------------")
print("Puntos de demanda evaluados:", len(demand))
print("SINR calculado para todos los puntos: OK")
print("Todos los SINR lineales son positivos: OK")
print("Todos los valores son finitos: OK")
print("Coincidencia con métricas del baseline: OK")
print()

print("SINR promedio:", avg_sinr_db_s2, "dB")
print("Percentil 5 del SINR:", p05_sinr_db_s2, "dB")


# ------------------------------------------------------------
# Mostrar algunos resultados
# ------------------------------------------------------------

sinr_results = pd.DataFrame({
    "demand_point": np.arange(len(demand)),
    "serving_site": serving_site_global_s2,
    "serving_rsrp_dbm": serving_rsrp_dbm_s2,
    "interference_w": interference_w_s2,
    "sinr_linear": sinr_linear_s2,
    "sinr_db": sinr_db_s2
})

sinr_results.head(10)

PB15 — Cálculo de SINR
----------------------------------
Puntos de demanda evaluados: 220
SINR calculado para todos los puntos: OK
Todos los SINR lineales son positivos: OK
Todos los valores son finitos: OK
Coincidencia con métricas del baseline: OK

SINR promedio: 7.907272 dB
Percentil 5 del SINR: -9.585938787460327 dB


,demand_point,serving_site,serving_rsrp_dbm,interference_w,sinr_linear,sinr_db
0,0,0,-75.471771,1.713305e-12,11.295491,10.529051
1,1,3,-67.153893,3.913064e-12,40.877285,16.114820
2,2,16,-91.421471,1.038675e-13,0.799208,-0.973403
3,3,3,-72.603554,5.451837e-12,8.785551,9.437690
4,4,3,-70.460770,4.583584e-12,16.711086,12.230047
5,5,3,-80.579575,2.229999e-12,2.889825,4.608716
6,6,3,-85.704315,1.186722e-12,1.354708,1.318457
7,7,3,-84.480751,8.147916e-13,2.209623,3.443182
8,8,3,-65.780739,1.180307e-10,2.223334,3.470047
9,9,3,-76.025604,6.557326e-12,3.394938,5.308319


## Conversión de SINR a eficiencia espectral

In [10]:
# ============================================================
# SOLUCIÓN PB17 — CÁLCULO Y VALIDACIÓN
# ============================================================

# 1. Calcular eficiencia espectral acotada al límite SE_CAP
se_bphz_s2 = np.minimum(np.log2(1 + sinr_linear_s2), SE_CAP)

# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS (Sanity Checks)
# ------------------------------------------------------------

# 1. Deben existir 220 valores (mismo tamaño que el SINR)
assert len(se_bphz_s2) == len(sinr_linear_s2), "El vector no tiene 220 valores"

# 2. Todos los valores deben ser >= 0
assert np.all(se_bphz_s2 >= 0), "Hay valores de eficiencia espectral negativos"

# 3. Ningún valor debe superar SE_CAP = 6 bps/Hz
assert np.all(se_bphz_s2 <= SE_CAP), "Hay valores que superan el límite SE_CAP"

# 4. Calcular el promedio de se_bphz_s2
avg_se_bphz = np.mean(se_bphz_s2)

# 5. Comparar dicho promedio con el reportado en el baseline
# Usamos np.isclose para evitar errores estrictos de redondeo de decimales
assert np.isclose(avg_se_bphz, baseline_metrics_s2["avg_se_bphz"]), "El cálculo difiere del baseline"

print("PB17 — Cálculo de Eficiencia Espectral")
print("--------------------------------------")
print("Eficiencia espectral calculada para todos los puntos: OK")
print("Límites de valores (>=0 y <=6): OK")
print("Coincidencia con métricas del baseline: OK")
print()
print(f"Eficiencia espectral promedio: {avg_se_bphz:.4f} bps/Hz")

PB17 — Cálculo de Eficiencia Espectral
--------------------------------------
Eficiencia espectral calculada para todos los puntos: OK
Límites de valores (>=0 y <=6): OK
Coincidencia con métricas del baseline: OK

Eficiencia espectral promedio: 2.8854 bps/Hz


## Demanda Busy Hour por punto

In [18]:
# ============================================================
# PB19 — Demanda Busy Hour por punto de demanda
# Responsable: Jhon
# ============================================================

# OBJETIVO:
# Calcular la demanda de tráfico en hora pico (Busy Hour)
# para cada uno de los puntos de demanda.
#
# Fórmula:
#
#       Demanda_BH =
#       subscribers_initial
#       × busy_hour_concurrency
#       × target_throughput
#
# ------------------------------------------------------------
# VARIABLES DISPONIBLES
# ------------------------------------------------------------
#
# demand["subscribers_initial"]
#
# params["demand"]["busy_hour_concurrency"]
#
# params["demand"]["target_throughput_mbps_per_subscriber"]
#
# ------------------------------------------------------------
# IMPORTANTE
# ------------------------------------------------------------
#
# El throughput objetivo viene expresado en Mbps,
# por lo que debe convertirse a bps multiplicando por:
#
#       1e6
#
# Se debe utilizar directamente:
#
#       demand["subscribers_initial"]
#
# NO volver a calcular el 30 % de households.
#
# En Sprint 1 se identificó que subscribers_initial fue
# calculado punto por punto y puede existir una diferencia
# frente a calcular el 30 % directamente sobre el total
# de hogares debido al redondeo por punto.
#
# ------------------------------------------------------------
# RESULTADO QUE DEBE DEJAR ESTA TAREA
# ------------------------------------------------------------
#
# Crear:
#
#       demand_bps_s2
#
# Debe contener la demanda Busy Hour, en bps,
# de cada uno de los 220 puntos.
#
# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS
# ------------------------------------------------------------
#
# 1. Deben existir 220 valores.
# 2. Todos los valores deben ser >= 0.
# 3. Verificar manualmente algunos puntos.
# 4. Comparar la fórmula utilizada con la implementada
#    dentro de evaluate_plan() en el baseline.
# ============================================================
# SOLUCIÓN PB19 — Demanda Busy Hour
# ============================================================

# 1. Extraer las variables de los diccionarios y dataframes
subs = demand["subscribers_initial"].to_numpy()
conc = params["demand"]["busy_hour_concurrency"]
target_mbps = params["demand"]["target_throughput_mbps_per_subscriber"]

# 2. Calcular la demanda total en bps (multiplicando por 1e6)
demand_bps_s2 = subs * conc * target_mbps * 1e6

# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS (Sanity Checks)
# ------------------------------------------------------------

# 1. Deben existir 220 valores
assert len(demand_bps_s2) == len(demand), "Error: No hay 220 valores"

# 2. Todos los valores deben ser >= 0
assert np.all(demand_bps_s2 >= 0), "Error: Hay valores de demanda negativos"

print("PB19 — Cálculo de Demanda Busy Hour")
print("--------------------------------------")
print("Demanda calculada para todos los puntos: OK")
print("Límites de valores (>=0): OK")
print()
# Mostrar los primeros 5 resultados como verificación manual
print("Muestra de demanda para los primeros 5 puntos (bps):")
print(demand_bps_s2[:5])

PB19 — Cálculo de Demanda Busy Hour
--------------------------------------
Demanda calculada para todos los puntos: OK
Límites de valores (>=0): OK

Muestra de demanda para los primeros 5 puntos (bps):
[16000000. 20000000. 16000000. 20000000. 16000000.]


## Capacidad y satisfacción de demanda por punto

In [19]:
# ============================================================
# PB18 — Capacidad y satisfacción de demanda por punto
# Responsable: Jhon
# ============================================================

# IMPORTANTE:
# Esta tarea debe realizarse después de completar PB17 y PB19.
#
# ------------------------------------------------------------
# OBJETIVO 1 — Calcular capacidad
# ------------------------------------------------------------
#
# Utilizar la eficiencia espectral calculada en PB17:
#
#       se_bphz_s2
#
# y el ancho de banda disponible:
#
#       BW_HZ
#
# Fórmula:
#
#       Capacidad = BW × SE
#
# Por lo tanto se debe obtener:
#
#       cap_bps_s2
#
# con la capacidad disponible para cada uno de los
# 220 puntos de demanda.
#
# ------------------------------------------------------------
# OBJETIVO 2 — Comparar capacidad contra demanda
# ------------------------------------------------------------
#
# Utilizar la demanda calculada en PB19:
#
#       demand_bps_s2
#
# Para cada punto verificar:
#
#       cap_bps_s2 >= demand_bps_s2
#
# Si se cumple:
#
#       True  -> la demanda del punto está satisfecha
#
# Si no se cumple:
#
#       False -> la capacidad disponible no satisface
#                la demanda Busy Hour del punto
#
# ------------------------------------------------------------
# RESULTADOS QUE DEBE DEJAR ESTA TAREA
# ------------------------------------------------------------
#
# Crear:
#
#       cap_bps_s2
#
#       satisfied_mask_s2
#
# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS
# ------------------------------------------------------------
#
# 1. cap_bps_s2 debe tener 220 valores.
# 2. Ninguna capacidad puede ser negativa.
# 3. satisfied_mask_s2 debe contener valores booleanos.
# 4. Debe existir un True/False por cada punto de demanda.
# 5. Comparar posteriormente el resultado agregado contra:
#
#       capacity_satisfied_frac
#
#    obtenido por evaluate_plan() en el baseline.
#
# NOTA:
# No modificar las variables desarrolladas por los otros
# integrantes. Utilizar los nombres definidos para facilitar
# la integración del notebook.

# ============================================================
# SOLUCIÓN PB18 — Capacidad y satisfacción de demanda por punto
# ============================================================

# OBJETIVO 1 — Calcular capacidad total (bps)
cap_bps_s2 = BW_HZ * se_bphz_s2

# OBJETIVO 2 — Comparar capacidad contra demanda (Máscara booleana)
satisfied_mask_s2 = cap_bps_s2 >= demand_bps_s2

# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS (Sanity Checks)
# ------------------------------------------------------------

# 1 y 2. Validar tamaño y que no haya capacidades negativas
assert len(cap_bps_s2) == len(demand), "Error: cap_bps_s2 no tiene 220 valores"
assert np.all(cap_bps_s2 >= 0), "Error: Hay capacidades calculadas negativas"

# 3 y 4. Validar máscara booleana
assert satisfied_mask_s2.dtype == bool, "Error: satisfied_mask_s2 no es tipo booleano"
assert len(satisfied_mask_s2) == len(demand), "Error: La máscara no tiene 220 valores"

# 5. Comparar el resultado agregado contra el baseline
# Extraemos los hogares como se hizo originalmente en evaluate_plan()
hh = demand["households"].to_numpy()

# Calculamos la fracción de capacidad satisfecha ponderada por hogares
capacity_satisfied_frac = float((hh * satisfied_mask_s2).sum() / hh.sum())

# Comparamos con las métricas del baseline (usamos np.isclose por los decimales)
assert np.isclose(capacity_satisfied_frac, baseline_metrics_s2["capacity_satisfied_frac"]), "Error: El porcentaje difiere del baseline"

print("PB18 — Capacidad y Satisfacción")
print("--------------------------------------")
print("Capacidad calculada (>=0 y 220 valores): OK")
print("Máscara de satisfacción (Booleana y 220 valores): OK")
print("Coincidencia con metrics del baseline: OK")
print()
print(f"Fracción de capacidad satisfecha general: {capacity_satisfied_frac:.4f} ({(capacity_satisfied_frac*100):.1f}%)")

PB18 — Capacidad y Satisfacción
--------------------------------------
Capacidad calculada (>=0 y 220 valores): OK
Máscara de satisfacción (Booleana y 220 valores): OK
Coincidencia con metrics del baseline: OK

Fracción de capacidad satisfecha general: 0.9796 (98.0%)


## Criterio de cobertura y comparación baseline

In [13]:
# ------------------------------------------------------------
# PB20 — Definir formalmente el criterio de cobertura
# ------------------------------------------------------------

# El scenario_params.json define un umbral mínimo de SINR
# para considerar que un punto se encuentra cubierto.
coverage_threshold_db_s2 = (
    params["radio"]["sinr_thresholds_db"]["coverage_min_sinr_db"]
)

# Un punto se considera cubierto cuando su SINR es mayor
# o igual al umbral definido en el JSON.
coverage_mask_s2 = sinr_db_s2 >= coverage_threshold_db_s2

# Número de puntos cubiertos
covered_points_s2 = np.sum(coverage_mask_s2)

# Hogares representados por cada punto de demanda
households_s2 = demand["households"].to_numpy()

# Total de hogares del escenario
total_households_s2 = households_s2.sum()

# Hogares ubicados en puntos que cumplen el umbral de cobertura
covered_households_s2 = households_s2[coverage_mask_s2].sum()

# Cobertura poblacional ponderada por hogares
pop_covered_frac_s2 = (
    covered_households_s2 / total_households_s2
)


# ------------------------------------------------------------
# PB21 — Comparación contra el criterio usado en el baseline
# ------------------------------------------------------------

# El baseline define cobertura mediante:
#
#       cap_bps > 0
#
# Para reproducir exactamente ese criterio utilizamos las mismas
# ecuaciones de eficiencia espectral y capacidad del baseline.

se_baseline_check_s2 = np.log2(1 + sinr_linear_s2)
se_baseline_check_s2 = np.minimum(
    se_baseline_check_s2,
    SE_CAP
)

cap_baseline_check_s2 = BW_HZ * se_baseline_check_s2

# Máscara de cobertura utilizada en el baseline
coverage_mask_baseline_s2 = cap_baseline_check_s2 > 0

# Cobertura poblacional según el criterio del baseline
pop_covered_frac_baseline_s2 = (
    households_s2[coverage_mask_baseline_s2].sum()
    / total_households_s2
)


# ------------------------------------------------------------
# Identificar los puntos que NO cumplen el criterio de SINR
# ------------------------------------------------------------

not_covered_points_s2 = np.where(~coverage_mask_s2)[0]

not_covered_households_s2 = (
    households_s2[~coverage_mask_s2].sum()
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

# Debe existir una decisión de cobertura por cada punto
assert len(coverage_mask_s2) == len(demand)

# La cobertura debe estar entre 0 y 1
assert 0 <= pop_covered_frac_s2 <= 1
assert 0 <= pop_covered_frac_baseline_s2 <= 1

# El total ponderado debe coincidir con el total de hogares
assert (
    covered_households_s2
    + not_covered_households_s2
    == total_households_s2
)

# El resultado calculado con el criterio del baseline debe coincidir
# con la métrica reportada originalmente por evaluate_plan()
baseline_metrics_pb21 = evaluate_plan(selected_sites_s2)

assert np.isclose(
    pop_covered_frac_baseline_s2,
    baseline_metrics_pb21["pop_covered_frac"]
)


# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print(" Cobertura poblacional")
print("----------------------------------------------")

print("Umbral de cobertura definido en JSON:",
      coverage_threshold_db_s2, "dB")

print()
print("CRITERIO FORMAL DEL SPRINT 2")
print("----------------------------------------------")
print("Puntos cubiertos:",
      covered_points_s2, "/", len(demand))

print("Puntos no cubiertos:",
      len(not_covered_points_s2), "/", len(demand))

print("Hogares cubiertos:",
      covered_households_s2, "/", total_households_s2)

print("Hogares no cubiertos:",
      not_covered_households_s2)

print("Cobertura poblacional:",
      f"{pop_covered_frac_s2:.4f}",
      f"({pop_covered_frac_s2*100:.2f} %)")

print()
print("CRITERIO UTILIZADO POR EL BASELINE")
print("----------------------------------------------")

print("Cobertura baseline (cap_bps > 0):",
      f"{pop_covered_frac_baseline_s2:.4f}",
      f"({pop_covered_frac_baseline_s2*100:.2f} %)")

print()
print("DIFERENCIA ENTRE CRITERIOS")
print("----------------------------------------------")

difference_coverage_s2 = (
    pop_covered_frac_baseline_s2
    - pop_covered_frac_s2
)

print("Diferencia:",
      f"{difference_coverage_s2:.4f}",
      f"({difference_coverage_s2*100:.2f} puntos porcentuales)")

print()
print("Puntos que no cumplen SINR >=", coverage_threshold_db_s2, "dB:")
print(not_covered_points_s2)


# ------------------------------------------------------------
# Tabla de puntos no cubiertos
# ------------------------------------------------------------

not_covered_results_s2 = pd.DataFrame({
    "demand_point": not_covered_points_s2,
    "households": demand.loc[
        not_covered_points_s2,
        "households"
    ].to_numpy(),
    "serving_site": serving_site_global_s2[
        not_covered_points_s2
    ],
    "serving_rsrp_dbm": serving_rsrp_dbm_s2[
        not_covered_points_s2
    ],
    "sinr_db": sinr_db_s2[
        not_covered_points_s2
    ]
})

not_covered_results_s2

 Cobertura poblacional
----------------------------------------------
Umbral de cobertura definido en JSON: -3.0 dB

CRITERIO FORMAL DEL SPRINT 2
----------------------------------------------
Puntos cubiertos: 198 / 220
Puntos no cubiertos: 22 / 220
Hogares cubiertos: 2300 / 2400
Hogares no cubiertos: 100
Cobertura poblacional: 0.9583 (95.83 %)

CRITERIO UTILIZADO POR EL BASELINE
----------------------------------------------
Cobertura baseline (cap_bps > 0): 1.0000 (100.00 %)

DIFERENCIA ENTRE CRITERIOS
----------------------------------------------
Diferencia: 0.0417 (4.17 puntos porcentuales)

Puntos que no cumplen SINR >= -3.0 dB:
[116 171 173 175 176 178 179 183 188 190 192 198 204 206 208 209 210 211
 212 214 215 219]


,demand_point,households,serving_site,serving_rsrp_dbm,sinr_db
0,116,15,7,-99.258507,-8.470938
1,171,6,13,-99.983559,-10.114977
2,173,6,0,-96.458549,-6.764797
3,175,4,6,-99.857964,-9.561555
4,176,3,13,-96.877159,-6.468086
5,178,6,16,-94.096756,-3.667595
6,179,7,3,-95.210762,-4.942649
7,183,7,13,-110.067001,-19.195044
8,188,4,11,-102.307098,-11.370012
9,190,6,11,-105.501518,-14.641828


## Implementación y verificación de métricas técnicas

In [14]:
# ============================================================
# PB22 — Implementación y verificación de métricas técnicas
# Responsable: Emanuel
# ============================================================

# IMPORTANTE:
# Esta tarea debe realizarse cuando estén integrados:
#
# PB13 -> asociación por mejor RSRP
# PB14 -> interferencia reuse-1
# PB15 -> SINR
# PB17 -> eficiencia espectral
# PB18 -> capacidad y satisfacción de demanda
# PB19 -> demanda Busy Hour
# PB20/PB21 -> cobertura corregida por umbral de SINR
#
# ------------------------------------------------------------
# OBJETIVO
# ------------------------------------------------------------
#
# Calcular e integrar las métricas técnicas obligatorias
# del Sprint 2 en una única estructura de resultados.
#
# Las métricas exigidas por el proyecto son:
#
#       pop_covered_frac
#       capacity_satisfied_frac
#       avg_sinr_db
#       p05_sinr_db
#       avg_se_bphz
#       sites_deployed
#
# ------------------------------------------------------------
# VARIABLES DISPONIBLES
# ------------------------------------------------------------
#
# Del bloque de radio:
#
#       sinr_db_s2
#
# Del bloque de cobertura:
#
#       coverage_mask_s2
#       pop_covered_frac_s2
#
# Del Integrante 2:
#
#       se_bphz_s2
#       cap_bps_s2
#       demand_bps_s2
#       satisfied_mask_s2
#
# Del dataset:
#
#       demand["households"]
#
# Del escenario evaluado:
#
#       selected_sites_s2
#
# ------------------------------------------------------------
# CÁLCULOS A REALIZAR
# ------------------------------------------------------------
#
# 1. Cobertura poblacional:
#
#       utilizar pop_covered_frac_s2
#
#    ya calculada en PB20 usando:
#
#       SINR >= coverage_min_sinr_db
#
#
# 2. Fracción de capacidad satisfecha:
#
#    Debe calcularse ponderando satisfied_mask_s2
#    con la cantidad de hogares de cada punto:
#
#       sum(households * satisfied_mask)
#       --------------------------------
#              sum(households)
#
#
# 3. SINR promedio:
#
#       promedio de sinr_db_s2
#
#
# 4. Percentil 5 del SINR:
#
#       quantile(sinr_db_s2, 0.05)
#
#
# 5. Eficiencia espectral promedio:
#
#       promedio de se_bphz_s2
#
#
# 6. Número de sitios desplegados:
#
#       len(selected_sites_s2)
#
# ------------------------------------------------------------
# RESULTADO QUE DEBE DEJAR ESTA TAREA
# ------------------------------------------------------------
#
# Crear un diccionario:
#
#       metrics_sprint2
#
# con una estructura similar a:
#
# metrics_sprint2 = {
#     "sites_deployed": ...,
#     "pop_covered_frac": ...,
#     "capacity_satisfied_frac": ...,
#     "avg_sinr_db": ...,
#     "p05_sinr_db": ...,
#     "avg_se_bphz": ...
# }
#
# ------------------------------------------------------------
# VALIDACIONES MÍNIMAS
# ------------------------------------------------------------
#
# 1. pop_covered_frac debe estar entre 0 y 1.
# 2. capacity_satisfied_frac debe estar entre 0 y 1.
# 3. sites_deployed debe ser <= max_sites_to_deploy.
# 4. Todos los valores numéricos deben ser finitos.
#
# IMPORTANTE:
# La cobertura de Sprint 2 NO debe reemplazarse por el
# criterio cap_bps > 0 utilizado originalmente por el baseline.
#
# Debe utilizarse el criterio corregido de PB20/PB21.

## Pruebas y sanity checks del modelo técnico completo

In [20]:
# ============================================================
# PB23 — Pruebas y sanity checks del modelo técnico completo
# Responsable: Jhon
# Revisión final: Todo el equipo
# ============================================================

# IMPORTANTE:
# Esta tarea se realiza cuando todas las variables principales
# del modelo técnico estén integradas.
#
# ------------------------------------------------------------
# OBJETIVO
# ------------------------------------------------------------
#
# Crear pruebas sencillas que permitan detectar errores de:
#
# - dimensiones
# - unidades
# - valores imposibles
# - lógica
# - integración entre tareas
#
# No se requieren pruebas extremadamente complejas.
# El objetivo es demostrar que el modelo fue validado
# manualmente y mediante sanity checks.
#
# ------------------------------------------------------------
# VARIABLES PRINCIPALES A REVISAR
# ------------------------------------------------------------
#
#       serving_site_global_s2
#       serving_rsrp_dbm_s2
#       interference_w_s2
#       sinr_linear_s2
#       sinr_db_s2
#       se_bphz_s2
#       demand_bps_s2
#       cap_bps_s2
#       coverage_mask_s2
#       satisfied_mask_s2
#       metrics_sprint2
#
# ------------------------------------------------------------
# PRUEBAS MÍNIMAS RECOMENDADAS
# ------------------------------------------------------------
#
# 1. Deben existir 220 resultados por punto para todas las
#    variables técnicas correspondientes.
#
# 2. Todos los sitios servidores deben pertenecer a:
#
#       selected_sites_s2
#
# 3. interference_w_s2 debe ser >= 0.
#
# 4. sinr_linear_s2 debe ser > 0.
#
# 5. Todos los valores de sinr_db_s2 deben ser finitos.
#
# 6. se_bphz_s2 debe cumplir:
#
#       0 <= SE <= 6
#
# 7. demand_bps_s2 debe ser >= 0.
#
# 8. cap_bps_s2 debe ser >= 0.
#
# 9. coverage_mask_s2 debe ser booleano.
#
# 10. satisfied_mask_s2 debe ser booleano.
#
# 11. Las fracciones de cobertura y capacidad satisfecha
#     deben encontrarse entre 0 y 1.
#
# 12. El número de sitios desplegados debe cumplir:
#
#       S <= max_sites_to_deploy
#
# ------------------------------------------------------------
# PRUEBA ESPECIAL RECOMENDADA
# ------------------------------------------------------------
#
# Evaluar manualmente uno o varios puntos de demanda y comprobar:
#
# RSRP servidor
#       ↓
# interferencia
#       ↓
# SINR
#       ↓
# eficiencia espectral
#       ↓
# capacidad
#       ↓
# demanda
#
# Esto permite demostrar que la cadena completa del modelo
# produce resultados coherentes.
#
# ------------------------------------------------------------
# RESULTADO ESPERADO
# ------------------------------------------------------------
#
# Crear assertions y mensajes de confirmación.
#
# Ejemplo:
#
#       assert ...
#       print("Sanity check X: OK")
#
# Al finalizar debe quedar claro que el modelo técnico completo
# fue verificado antes de utilizarse en los siguientes sprints.
# ============================================================
# SOLUCIÓN PB23 — Pruebas y sanity checks del modelo técnico
# ============================================================

print("Iniciando pruebas de cordura (Sanity Checks)...\n")

# 1. Dimensiones (Deben existir 220 resultados por variable)
N = len(demand)
assert len(serving_site_global_s2) == N
assert len(serving_rsrp_dbm_s2) == N
assert len(interference_w_s2) == N
assert len(sinr_linear_s2) == N
assert len(sinr_db_s2) == N
assert len(se_bphz_s2) == N
assert len(demand_bps_s2) == N
assert len(cap_bps_s2) == N
assert len(coverage_mask_s2) == N
assert len(satisfied_mask_s2) == N
print("1. Dimensiones de vectores correctas (220 valores): OK")

# 2. Sitios servidores deben pertenecer a selected_sites_s2
assert set(serving_site_global_s2).issubset(set(selected_sites_s2))
print("2. Pertenencia de sitios servidores validada: OK")

# 3. interference_w_s2 >= 0
assert np.all(interference_w_s2 >= 0)
print("3. Valores de interferencia >= 0: OK")

# 4. sinr_linear_s2 > 0
assert np.all(sinr_linear_s2 > 0)
print("4. Valores de SINR lineal > 0: OK")

# 5. Valores finitos en sinr_db_s2
assert np.all(np.isfinite(sinr_db_s2))
print("5. Valores de SINR en dB son finitos: OK")

# 6. se_bphz_s2 debe cumplir 0 <= SE <= 6 (SE_CAP)
assert np.all((se_bphz_s2 >= 0) & (se_bphz_s2 <= SE_CAP))
print("6. Eficiencia espectral dentro de los límites: OK")

# 7 y 8. Demandas y capacidades >= 0
assert np.all(demand_bps_s2 >= 0)
assert np.all(cap_bps_s2 >= 0)
print("7 y 8. Capacidades y demandas calculadas >= 0: OK")

# 9 y 10. Máscaras de tipo booleano (True/False)
assert coverage_mask_s2.dtype == bool
assert satisfied_mask_s2.dtype == bool
print("9 y 10. Máscaras convertidas correctamente a booleanos: OK")

# 11. Fracciones entre 0 y 1
cov_frac = baseline_metrics_s2["pop_covered_frac"]
cap_frac = baseline_metrics_s2["capacity_satisfied_frac"]
assert 0.0 <= cov_frac <= 1.0
assert 0.0 <= cap_frac <= 1.0
print("11. Fracciones de desempeño de la red acotadas [0,1]: OK")

# 12. Sitios desplegados respeta el máximo permitido
max_sites = params["planning"]["max_sites_to_deploy"]
assert len(selected_sites_s2) <= max_sites
print("12. Regla de negocio (max sitios desplegados) cumplida: OK\n")

# ------------------------------------------------------------
# PRUEBA ESPECIAL RECOMENDADA (Inspección manual punto 0)
# ------------------------------------------------------------
pto = 0
print(f"--- Trazabilidad de la red para el Punto de Demanda {pto} ---")
print(f"Señal (RSRP servidor): {serving_rsrp_dbm_s2[pto]:.2f} dBm")
print(f"Interferencia (W):     {interference_w_s2[pto]:.3e} Watts")
print(f"SINR calculado (dB):   {sinr_db_s2[pto]:.2f} dB")
print(f"Eficiencia espectral:  {se_bphz_s2[pto]:.2f} bps/Hz")
print(f"Capacidad entregada:   {cap_bps_s2[pto]/1e6:.2f} Mbps")
print(f"Demanda exigida:       {demand_bps_s2[pto]/1e6:.2f} Mbps")
print(f"¿Cliente satisfecho?:  {'Sí (True)' if satisfied_mask_s2[pto] else 'No (False)'}")
print("============================================================\n")
print("¡TODOS LOS SANITY CHECKS DEL MODELO TÉCNICO PASARON CON ÉXITO!")



Iniciando pruebas de cordura (Sanity Checks)...

1. Dimensiones de vectores correctas (220 valores): OK
2. Pertenencia de sitios servidores validada: OK
3. Valores de interferencia >= 0: OK
4. Valores de SINR lineal > 0: OK
5. Valores de SINR en dB son finitos: OK
6. Eficiencia espectral dentro de los límites: OK
7 y 8. Capacidades y demandas calculadas >= 0: OK
9 y 10. Máscaras convertidas correctamente a booleanos: OK
11. Fracciones de desempeño de la red acotadas [0,1]: OK
12. Regla de negocio (max sitios desplegados) cumplida: OK

--- Trazabilidad de la red para el Punto de Demanda 0 ---
Señal (RSRP servidor): -75.47 dBm
Interferencia (W):     1.713e-12 Watts
SINR calculado (dB):   10.53 dB
Eficiencia espectral:  3.62 bps/Hz
Capacidad entregada:   144.80 Mbps
Demanda exigida:       16.00 Mbps
¿Cliente satisfecho?:  Sí (True)

¡TODOS LOS SANITY CHECKS DEL MODELO TÉCNICO PASARON CON ÉXITO!


## Visualización de puntos de demanda y sitios candidatos

In [16]:
# ============================================================
# PB24 — Visualización de puntos de demanda y sitios candidatos
# Responsable: Emanuel
# ============================================================

# ESTA TAREA PUEDE REALIZARSE DESDE YA.
#
# No depende de PB17, PB18 o PB19.
#
# ------------------------------------------------------------
# OBJETIVO
# ------------------------------------------------------------
#
# Crear una visualización XY del escenario que permita observar:
#
# 1. Los 220 puntos de demanda.
# 2. Los 36 sitios candidatos.
# 3. Los sitios finalmente desplegados en selected_sites_s2.
#
# ------------------------------------------------------------
# VARIABLES DISPONIBLES
# ------------------------------------------------------------
#
# demand:
#
#       demand["x_km"]
#       demand["y_km"]
#
# sites:
#
#       sites["x_km"]
#       sites["y_km"]
#
# Sitios desplegados:
#
#       selected_sites_s2
#
# ------------------------------------------------------------
# VISUALIZACIÓN RECOMENDADA
# ------------------------------------------------------------
#
# Utilizar matplotlib.
#
# Mostrar:
#
# - puntos de demanda mediante scatter
# - sitios candidatos con un marcador diferente
# - sitios desplegados claramente diferenciados
#
# Agregar:
#
# - título
# - etiquetas x [km] y y [km]
# - leyenda
# - grid
#
# IMPORTANTE:
# Las coordenadas son coordenadas locales del escenario
# expresadas en kilómetros.
#
# No es necesario utilizar mapas reales ni cartografía externa.
#
# ------------------------------------------------------------
# RESULTADO ESPERADO
# ------------------------------------------------------------
#
# Una gráfica clara y sencilla que pueda utilizarse durante
# la sustentación para explicar la distribución espacial
# de demanda y la selección de sitios.

## Visualizaciones técnicas de SINR, cobertura y capacidad

In [17]:
# ============================================================
# PB25 — Visualizaciones técnicas de SINR, cobertura y capacidad
# Responsable: Emanuel
# ============================================================

# IMPORTANTE:
# Esta tarea debe realizarse cuando estén integradas las
# variables finales del modelo técnico.
#
# ------------------------------------------------------------
# OBJETIVO
# ------------------------------------------------------------
#
# Crear visualizaciones que permitan interpretar los resultados
# técnicos del Sprint 2.
#
# No agregar gráficas únicamente por decorar.
# Cada gráfica debe ayudar a explicar un resultado.
#
# ------------------------------------------------------------
# VARIABLES DISPONIBLES
# ------------------------------------------------------------
#
# Radio:
#
#       sinr_db_s2
#
# Cobertura:
#
#       coverage_mask_s2
#
# Capacidad:
#
#       cap_bps_s2
#
# Demanda:
#
#       demand_bps_s2
#
# Satisfacción:
#
#       satisfied_mask_s2
#
# Coordenadas:
#
#       demand["x_km"]
#       demand["y_km"]
#
# ------------------------------------------------------------
# VISUALIZACIONES MÍNIMAS SUGERIDAS
# ------------------------------------------------------------
#
# 1. DISTRIBUCIÓN DEL SINR
#
# Puede utilizarse un histograma del SINR en dB.
#
# Debe permitir observar:
#
# - distribución general
# - existencia de puntos con SINR bajo
# - relación con avg_sinr_db y p05_sinr_db
#
#
# 2. MAPA DE COBERTURA
#
# Utilizar las coordenadas XY para diferenciar:
#
#       coverage_mask_s2 == True
#       coverage_mask_s2 == False
#
# Esto permitirá observar espacialmente dónde se encuentran
# los puntos que no alcanzan el umbral de SINR.
#
#
# 3. CAPACIDAD VS DEMANDA
#
# Comparar:
#
#       cap_bps_s2
#
# contra:
#
#       demand_bps_s2
#
# Puede utilizarse una gráfica de dispersión u otra
# representación clara.
#
# Debe permitir identificar los puntos donde:
#
#       capacidad < demanda
#
#
# ------------------------------------------------------------
# RESULTADO ESPERADO
# ------------------------------------------------------------
#
# Crear aproximadamente 2 o 3 gráficas útiles.
#
# Cada gráfica debe incluir:
#
# - título
# - nombres de ejes
# - unidades
# - leyenda cuando corresponda
#
# IMPORTANTE:
# Dejar las figuras listas para ser utilizadas posteriormente
# en el informe y/o presentación del proyecto.

## Documentación de resultados y validación del Sprint 2
### Responsable: Emanuel, Tarea: PB26 
### Revisión final: Todo el equipo

Esta sección debe completarse únicamente cuando las tareas PB22, PB23, PB24 y PB25 estén terminadas y los resultados del equipo se encuentren integrados.

### 1. Configuración evaluada

Registrar:

- Sitios desplegados: `selected_sites_s2`
- Número de sitios desplegados:
- Esquema de interferencia: frequency reuse-1
- Ancho de banda:
- Umbral mínimo de cobertura SINR:

### 2. Métricas finales del Sprint 2

Reportar los valores almacenados en `metrics_sprint2`:

- `sites_deployed`:
- `pop_covered_frac`:
- `capacity_satisfied_frac`:
- `avg_sinr_db`:
- `p05_sinr_db`:
- `avg_se_bphz`:

### 3. Comparación con el baseline

Indicar cuáles resultados fueron validados y coincidieron con el modelo base:

- Asociación por mejor RSRP.
- Interferencia reuse-1.
- Ruido térmico.
- SINR.
- Eficiencia espectral.
- Capacidad y demanda Busy Hour.

Documentar también las diferencias encontradas.

En particular, PB20 y PB21 identificaron una diferencia en la definición de cobertura:

- El baseline utiliza `cap_bps > 0`.
- `scenario_params.json` define cobertura mediante el umbral `coverage_min_sinr_db`.

Por esta razón, la cobertura poblacional obtenida en Sprint 2 puede diferir de la reportada originalmente por el baseline.

### 4. Validaciones realizadas

Resumir las verificaciones desarrolladas durante el sprint:

- Sanity checks.
- Comparación contra resultados del baseline.
- Verificación manual de algunos puntos de demanda.
- Revisión de unidades.
- Inconsistencias encontradas y correcciones realizadas.

### 5. Interpretación técnica

Explicar brevemente:

- Comportamiento general del SINR.
- Comportamiento de los puntos con peor SINR.
- Efecto de la interferencia reuse-1.
- Cobertura poblacional obtenida.
- Fracción de demanda satisfecha.
- Relación entre calidad de señal y capacidad.

### 6. Conclusión del Sprint 2

Indicar si el modelo técnico quedó correctamente implementado y validado para ser utilizado como base del Sprint 3.

El Sprint 3 utilizará este modelo para desarrollar y comparar la primera estrategia de optimización de selección de sitios.